# Uganda Road VCI/PCI Estimator — Colab Training

**What this notebook does:**
1. Mount Google Drive and load pre-extracted Uganda features (~500 MB)
2. Download RDD2022 road damage dataset directly from FigShare (13.3 GB)
3. Pre-train the PCI head on RDD2022 (gives accurate PCI estimates)
4. Train all three heads (Defect + vVCI + PCI) on Uganda features
5. Save the final checkpoint to Google Drive

**Before running:** Upload these two files to your Google Drive:
- `outputs/features.npz` (from `scripts/extract_features.py`)
- `outputs/dataset.csv` (from `scripts/prepare_data.py`)

**Runtime:** GPU → T4 or better. Expected time: ~30 min.

In [ ]:
# ── 0. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name   :', torch.cuda.get_device_name(0))
    print('VRAM (GB)  :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: no GPU detected — change Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# ── EDIT THESE PATHS to match where you put the files in your Drive ──────────
DRIVE_ROOT    = '/content/drive/MyDrive/vci_estimator'
FEATURES_PATH = f'{DRIVE_ROOT}/features.npz'
DATASET_PATH  = f'{DRIVE_ROOT}/dataset.csv'
OUTPUT_DIR    = f'{DRIVE_ROOT}/outputs'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

assert os.path.exists(FEATURES_PATH), f'features.npz not found at {FEATURES_PATH}'
assert os.path.exists(DATASET_PATH),  f'dataset.csv not found at {DATASET_PATH}'
print('Drive mounted. Files found.')

In [ ]:
# ── 2. Install dependencies + clone project scripts ──────────────────────────
!pip install timm -q

# Copy project code from Drive (upload the whole project zip, or paste scripts inline)
# Option A: project zip in Drive
PROJECT_ZIP = f'{DRIVE_ROOT}/vci_estimator_scripts.zip'
if os.path.exists(PROJECT_ZIP):
    !unzip -q {PROJECT_ZIP} -d /content/project
    import sys; sys.path.insert(0, '/content/project')
    print('Project scripts loaded from zip.')
else:
    print('No zip found — will use inline implementations below.')

import numpy as np, pandas as pd
print('numpy:', np.__version__, '| pandas:', pd.__version__)

In [ ]:
# ── 3. Inspect features + dataset ────────────────────────────────────────────
import numpy as np, pandas as pd

npz = np.load(FEATURES_PATH, allow_pickle=True)
feats = npz['features']   # (N, 1536) float16
paths = npz['image_paths']
print(f'Features shape : {feats.shape}  dtype={feats.dtype}')
print(f'File size      : {os.path.getsize(FEATURES_PATH)/1e6:.0f} MB')

df = pd.read_csv(DATASET_PATH)
print(f'\nDataset rows   : {len(df):,}')
print(f'Splits         : {df["split"].value_counts().to_dict()}')
print(f'Roads          : {df["road_name"].nunique()}')
print(f'Survey years   : {sorted(df["survey_year"].dropna().unique().astype(int).tolist())}')
print(f'vVCI range     : {df["vvci"].min():.1f} – {df["vvci"].max():.1f}  mean={df["vvci"].mean():.1f}')

In [ ]:
# ── 4. Download RDD2022 from FigShare ─────────────────────────────────────────
# RDD2022 is publicly available. We use Japan subset (highest quality annotations).
# Full dataset is 13.3 GB; Japan alone is ~3 GB — enough for PCI pre-training.
import os

RDD_DIR = '/content/rdd2022'
os.makedirs(RDD_DIR, exist_ok=True)

# Check if already downloaded to Drive (saves re-downloading)
RDD_DRIVE_CACHE = f'{DRIVE_ROOT}/rdd2022_Japan.zip'

if os.path.exists(RDD_DRIVE_CACHE):
    print('Using cached RDD2022 Japan from Drive ...')
    !unzip -q {RDD_DRIVE_CACHE} -d {RDD_DIR}
else:
    print('Downloading RDD2022 Japan from FigShare (~3 GB) ...')
    # FigShare DOI: 10.6084/m9.figshare.21504930
    # Japan subset URL (direct download)
    JAPAN_URL = 'https://figshare.com/ndownloader/files/38346750'
    !wget -q --show-progress -O /content/rdd2022_Japan.zip "{JAPAN_URL}"
    !unzip -q /content/rdd2022_Japan.zip -d {RDD_DIR}
    # Cache to Drive so we don't re-download next session
    !cp /content/rdd2022_Japan.zip {RDD_DRIVE_CACHE}
    print('Cached to Drive for future sessions.')

# Verify
import glob
japan_imgs = glob.glob(f'{RDD_DIR}/**/images/*.jpg', recursive=True)
japan_xmls = glob.glob(f'{RDD_DIR}/**/annotations/xml/*.xml', recursive=True)
print(f'RDD2022 Japan: {len(japan_imgs):,} images, {len(japan_xmls):,} annotations')

In [ ]:
# ── 5. Pre-train PCI head on RDD2022 ─────────────────────────────────────────
# The PCI head sees international road damage images with ASTM-derived PCI labels
# before it ever sees Uganda data. This gives it a much better initialisation.

PCI_PRETRAIN_PATH = f'{OUTPUT_DIR}/pci_head.pt'

if os.path.exists(PCI_PRETRAIN_PATH):
    print(f'Pre-trained PCI head found at {PCI_PRETRAIN_PATH} — skipping pre-training.')
    print('Delete the file and re-run this cell to force re-training.')
else:
    !python /content/project/scripts/pretrain_pci.py \
        --data        {RDD_DIR} \
        --countries   Japan \
        --output-dir  {OUTPUT_DIR}/pci_pretrain \
        --device      cuda \
        --epochs-stage1 10 \
        --epochs-stage2 10 \
        --batch-size  64
    
    import shutil
    shutil.copy(f'{OUTPUT_DIR}/pci_pretrain/pci_head.pt', PCI_PRETRAIN_PATH)
    print(f'Pre-trained PCI head saved → {PCI_PRETRAIN_PATH}')

In [ ]:
# ── 6. Train all heads on Uganda features ────────────────────────────────────
# batch-size 512 fits easily on T4 since there is no backbone forward pass.
# Training 40 epochs should take ~15–25 minutes on a T4.

!python /content/project/scripts/train_features.py \
    --features      {FEATURES_PATH} \
    --dataset       {DATASET_PATH} \
    --output-dir    {OUTPUT_DIR} \
    --device        cuda \
    --epochs        40 \
    --batch-size    512 \
    --pci-pretrain  {PCI_PRETRAIN_PATH}

In [ ]:
# ── 7. Plot training curves ───────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = f'{OUTPUT_DIR}/metrics.csv'
if os.path.exists(metrics_path):
    m = pd.read_csv(metrics_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(m['epoch'], m['train_loss'], label='train')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(m['epoch'], m['val_mae_vvci'], label='val', color='orange')
    axes[1].set_title('Val MAE (vVCI)'); axes[1].set_xlabel('Epoch')

    axes[2].plot(m['epoch'], m['val_acc_defect_mean'], label='val', color='green')
    axes[2].set_title('Defect Accuracy'); axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=120)
    plt.show()
    print(f'\nBest val MAE: {m["val_mae_vvci"].min():.2f} at epoch {m["val_mae_vvci"].idxmin()}')
else:
    print('metrics.csv not found — training may not have completed')

In [ ]:
# ── 8. Verify checkpoint + print final metrics ────────────────────────────────
import torch

ckpt_path = f'{OUTPUT_DIR}/checkpoints/best.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    print(f'Checkpoint epoch  : {ckpt["epoch"]}')
    print(f'Best val MAE      : {ckpt["val_mae"]:.2f}')
    print(f'Feature-based     : {ckpt.get("feature_based", False)}')
    print(f'Checkpoint size   : {os.path.getsize(ckpt_path)/1e6:.1f} MB')
    print(f'\nSaved to Drive at : {ckpt_path}')
else:
    print('best.pt not found')

## After training

Download `best.pt` from your Drive and place it at `outputs/checkpoints/best.pt` on your local machine.

**Note**: Because training used pre-extracted features (no backbone), `best.pt` contains only the three head weights. To run the full inference pipeline (which needs the backbone), the `api/inference.py` must be updated to either:
1. Load the full EfficientNet-B3 backbone + attach the trained head weights
2. Or export using `scripts/export_features_model.py` (to be written)

The `evaluate.py` script will work as-is if you point it at `dataset.csv` and use the feature-based inference path.